In [6]:
# 환경설정
import os
import sys
from tqdm import tqdm
# duckdb 활용
import duckdb

# langchain 라이브러리 활용
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter


# es strategy 전략
from langchain_elasticsearch import DenseVectorStrategy #   query vector 전략
from langchain_elasticsearch import BM25Strategy # 키워드 기반 전략

from uuid import uuid4

import pandas as pd
import numpy as np

# chain
from langchain.chains import RetrievalQA

from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableMap
from langchain_core.prompts import ChatPromptTemplate


# 환경변수 세팅
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:

# api key
if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = ''

# Ebedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Es 구축 - bm25 ~ 키워드 기반 VectorDB
vector_store = ElasticsearchStore(
    es_url="https://my-elasticsearch-project-fbc592.es.us-east-1.aws.elastic.cloud:443",
    index_name="slu_llm",
    embedding=embeddings,
    es_api_key= 'V01OZzlKY0JfU1N6SDBoNGVuck86VmVnbE1EUzR2NEVBTFJOTjZSRWx0dw==',
    strategy=DenseVectorStrategy()
)

In [8]:
news_conn = duckdb.connect('../DB/news.db')
news_conn.close()
ETF_conn = duckdb.connect('../DB/ETF.db')
ETF_conn.close()

In [10]:
#target_df = news_conn.execute('select * from 거시경제_table').fetchdf()
target_df = pd.read_csv('../data/삼성전자_target_news.csv')
print(target_df.content.str.contains('삼성전자').sum())

444


In [19]:
target_df = target_df.dropna()

In [20]:
target_df.content.values

array(['네이버의 가상 콘텐츠 제작 스튜디오 비전·모션스테이지에서 광학식 슈트를 착용한 직원이 3차원(3D) 애니메이션 구현을 시연하고 있다. 스튜디오에 설치된 적외선 카메라와 슈트에 장착된 전자 센서가 손가락 움직임과 얼굴 표정까지 정밀하게 포착해 캐릭터를 실시간으로 만들어낸다. \ufeff /네이버 제공\n\n                네이버가 확장현실(XR) 콘텐츠 시장에 본격 참전한다. 삼성전자가 연내 선보일 첫 XR 기기 ‘프로젝트 무한’에 담을 콘텐츠를 생산하는 것이 우선 목표다. 웹(컴퓨터)에서 모바일(스마트폰)로의 전환 과정에서 성공적으로 살아남은 네이버가 차세대 하드웨어 시장에서도 소프트웨어와 콘텐츠를 중심으로 생존력을 증명하려는 시도로 풀이된다.네이버는 경기 성남시 네이버 사옥에서 지난 16일 ‘이머시브 미디어 플랫폼 테크 포럼’을 열었다. 언론을 대상으로 XR 플랫폼 전략과 차세대 미디어 기술을 공개하기 위한 자리다. 17일 네이버 관계자는 “생성형 인공지능(AI)과 실시간 3차원(3D) 렌더링 기술을 기반으로 가상현실(VR), 증강현실(AR), 혼합현실(MR)을 아우르는 XR 콘텐츠 생태계를 구축하는 것이 최종 목표”라고 설명했다.\n\n\n\n\n\n                스마트글라스와 VR 헤드셋으로 대표되는 XR 기기는 스마트폰을 잇는 새로운 하드웨어 플랫폼으로 부상하고 있다. 메타는 2019년 첫 VR 헤드셋 ‘퀘스트’를 출시했다. 당시 마크 저커버그 메타 최고경영자(CEO)는 VR 헤드셋을 ‘차세대 컴퓨팅 플랫폼’이라고 선언했다. 지난해 10월에는 AI 기능을 적용한 ‘퀘스트3S’를 선보였다.구글은 삼성전자, 젠틀몬스터와 협업해 안드로이드 기반 XR 기기를 내놓을 예정이다. 삼성은 구글, 퀄컴과 협업한 또 다른 XR 제품도 준비 중이다. 노태문 삼성전자 사장은 9일 뉴욕에서 열린 기자간담회에서 “연내 XR 헤드셋 출시를 목표로 완성도를 높이고 있다”고 밝혔다.네이버는 삼성의 신제품 출시에 발맞춰 XR 기반의 콘텐츠 플

In [23]:
docs = [Document(page_content=doc,metadata={'source':'삼성전자'}) for doc in target_df.content.values]

In [63]:
# 1. Text splitter 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=500,
)

chunked_docs = []

for content in target_df.content.values:
    # 각 기사 고유 ID (모든 chunk에 동일하게 넣을 ID)
    article_id = str(uuid4())

    # 원본 기사 문서
    original_doc = Document(page_content=content, metadata={"source": "삼성전자", "article_id": article_id})

    # chunk로 나눔
    chunks = text_splitter.split_documents([original_doc])

    # 각 chunk에도 같은 article_id 포함됨
    chunked_docs.extend(chunks)

# vector_store에 넣기 위한 UUID (각 chunk는 unique해야 하므로 다시 생성)
uuids = [str(uuid4()) for _ in range(len(chunked_docs))]


In [64]:
[x.page_content for x in chunked_docs if '삼성전자' in x.page_content]

['네이버의 가상 콘텐츠 제작 스튜디오 비전·모션스테이지에서 광학식 슈트를 착용한 직원이 3차원(3D) 애니메이션 구현을 시연하고 있다. 스튜디오에 설치된 적외선 카메라와 슈트에 장착된 전자 센서가 손가락 움직임과 얼굴 표정까지 정밀하게 포착해 캐릭터를 실시간으로 만들어낸다. \ufeff /네이버 제공\n\n                네이버가 확장현실(XR) 콘텐츠 시장에 본격 참전한다. 삼성전자가 연내 선보일 첫 XR 기기 ‘프로젝트 무한’에 담을 콘텐츠를 생산하는 것이 우선 목표다. 웹(컴퓨터)에서 모바일(스마트폰)로의 전환 과정에서 성공적으로 살아남은 네이버가 차세대 하드웨어 시장에서도 소프트웨어와 콘텐츠를 중심으로 생존력을 증명하려는 시도로 풀이된다.네이버는 경기 성남시 네이버 사옥에서 지난 16일 ‘이머시브 미디어 플랫폼 테크 포럼’을 열었다. 언론을 대상으로 XR 플랫폼 전략과 차세대 미디어 기술을 공개하기 위한 자리다. 17일 네이버 관계자는 “생성형 인공지능(AI)과 실시간 3차원(3D) 렌더링 기술을 기반으로 가상현실(VR), 증강현실(AR), 혼합현실(MR)을 아우르는 XR 콘텐츠 생태계를 구축하는 것이 최종 목표”라고 설명했다.\n\n\n\n\n\n                스마트글라스와 VR 헤드셋으로 대표되는 XR 기기는 스마트폰을 잇는 새로운 하드웨어 플랫폼으로 부상하고 있다. 메타는 2019년 첫 VR 헤드셋 ‘퀘스트’를 출시했다. 당시 마크 저커버그 메타 최고경영자(CEO)는 VR 헤드셋을 ‘차세대 컴퓨팅 플랫폼’이라고 선언했다. 지난해 10월에는 AI 기능을 적용한 ‘퀘스트3S’를 선보였다.구글은 삼성전자, 젠틀몬스터와 협업해 안드로이드 기반 XR 기기를 내놓을 예정이다. 삼성은 구글, 퀄컴과 협업한 또 다른 XR 제품도 준비 중이다. 노태문 삼성전자 사장은 9일 뉴욕에서 열린 기자간담회에서 “연내 XR 헤드셋 출시를 목표로 완성도를 높이고 있다”고 밝혔다.네이버는 삼성의 신제품 출시에 발맞춰 XR 기반의 콘텐츠 플랫폼을 구축

In [65]:
print([len(doc.page_content) for doc in chunked_docs[:500]])

[1683, 1317, 749, 1047, 1125, 635, 1120, 3154, 1766, 1585, 2083, 561, 940, 1093, 970, 4106, 900, 1385, 930, 1881, 1972, 1678, 2581, 2722, 1228, 3028, 881, 1217, 1123, 1097, 1990, 800, 2743, 1613, 3952, 4998, 1810, 2001, 2021, 2409, 1394, 883, 2210, 2190, 2232, 770, 1943, 2321, 950, 1575, 2993, 1210, 844, 1989, 1999, 909, 1158, 3430, 1255, 682, 2512, 1230, 1493, 991, 1695, 1121, 2406, 1496, 1672, 1099, 1093, 1189, 1382, 1486, 2066, 1636, 570, 3700, 1182, 476, 876, 1253, 989, 1707, 1666, 714, 1187, 1572, 1029, 1803, 2044, 1598, 1995, 1301, 2083, 2105, 1336, 984, 1589, 1812, 1834, 1731, 2887, 2777, 1825, 1008, 802, 752, 1120, 1990, 947, 876, 2117, 1397, 1523, 1252, 932, 545, 2127, 1438, 1233, 1365, 1828, 1168, 700, 1989, 1571, 1005, 1780, 2099, 2347, 2218, 1224, 1270, 1674, 1357, 1956, 2782, 953, 1962, 1793, 953, 1342, 2567, 865, 2181, 1207, 1425, 1684, 1243, 1410, 1527, 1320, 2244, 1214, 787, 810, 2064, 802, 2113, 1255, 1507, 763, 1146, 1258, 576, 2232, 2498, 1154, 1080, 3113, 1406, 2105

In [66]:
batch_size = 10
for i in tqdm(range(0, len(chunked_docs), batch_size)):
    batch_docs = chunked_docs[i:i+batch_size]
    batch_ids = uuids[i:i+batch_size]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

100%|██████████| 18/18 [00:45<00:00,  2.55s/it]


In [67]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
)

In [68]:
retriever = vector_store.as_retriever(
    search_type="mmr",  # 또는 "similarity"
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

In [69]:
retriever.invoke('삼성전자')

[Document(metadata={'source': '삼성전자', 'article_id': '09eca0a8-667b-4752-8624-6e08b8a190b2'}, page_content="삼성전자 서초사옥 / 사진=한경DB\n\n                삼성전자가 스마트폰의 눈 역할을 하는 반도체 ‘이미지센서’ 신제품을 중국 샤오미에 공급했다.어두운 곳에서도 선명한 사진을 찍을 수 있는 ‘나노프리즘’ 기술을 앞세워 계약을 따냈다.이미지센서의 적용처가 자율주행차, 로봇 등으로 확대하며 관련 시장은 2029년 36조원까지 커질 것으로 전망된다. 삼성전자는 맞춤형 제품 개발을 통해 고객사를 북미 대형 정보기술(IT) 기업 등으로 확대하며 시장 공략에 속도를 낼 계획이다. 1년 만에 이미지센서 신제품 출시12일 삼성전자에 따르면 반도체 개발을 담당하는 시스템LSI사업부는 올 2분기 이미지센서 신제품 아이소셀 JNP를 개발해 양산을 시작했다. 아이소셀 JNP는 지난해 6월 아이소셀 HP9 등 프리미엄 이미지센서 3종을 내놓은 이후 약 1년 만에 선보인 신제품이다. 이미지센서는 카메라 렌즈를 통해 들어온 빛을 디지털 전기신호로 바꾸는 반도체로 스마트폰 등 IT 제품의 ‘눈’ 역할을 한다.삼성전자 아이소셀 JNP의 첫 외부 고객은 중국 스마트폰업체 중국 샤오미다. 샤오미는 최고의 카메라를 적용하고 있는 스마트폰 시리즈 시비(CIVI)의 신제품 CIVI 5 PRO에 아이소셀 JNP를 적용했다. 초격차 기술력 회복 시동아이소셀 JNP는 화소(픽셀) 크기 0.64마이크로미터(㎛, 1㎛=100만분의 1m), 화소 수 5000만, 옵티컬포맷(이미지센서 크기를 나타내는 단위) ‘1/2.8인치’로 겉으론 경쟁사 제품과 다를 바 없는 평범한 이미지센서처럼 보인다. 샤오미의 마음을 잡아끈 건 삼성전자가 업계 최초로 아이소셀 JNP에 적용한 나노프리즘 기술이다.최근 반도체업계에선 이미지센서를 작게 만드는 경쟁이 치열하다. IT 제품을 얇고 가볍게 만드는 추세가 이어진 결과다. 이 과

In [70]:
docs = retriever.get_relevant_documents('삼성전자 시황')
for doc in docs:
    print(doc.page_content)

서울 서초동 삼성전자 사옥/이솔 기자

                삼성전자가 지난해 9월 4일 이후 약 11개월 만에 '7만전자' 회복을 눈앞에 뒀다. 증권가에선 반도체주 투자와 관련해 SK하이닉스를 팔고(쇼트), 삼성전자를 사는(롱) 전략을 구사해볼 필요가 있다는 조언이 나온다. 21일 삼성전자는 유가증권시장에서 700원(1.04%) 오른 6만 7800원에 거래를 마쳤다. 최근 5거래일(7월 15일~21일) 동안 8.48% 상승했다. 삼성전자가 6만 원 후반대에 안착하자 투자자 상당수가 손실에서 벗어나 상승 구간에 진입했다. 네이버페이 '내자산‘ 서비스에 따르면 삼성전자 투자자 26만 5371명의 평균 매수 단가는 6만 7169원으로 집계됐다. 이들의 평균 수익률은 0.94%다.삼성전자는 7월 들어 13.38% 올랐다. 같은 기간 코스피 상승률(4.53%)을 8.85%포인트(p) 웃도는 상승률이다. 외국인들이 이달 들어 삼성전자 2조 701억 원어치 사들이면서 상승세를 견인했다. 삼성전자는 7월 외국인 순매수 1위 종목에 이름을 올렸다.
                    



삼성전자 주가 그래프/구글 캡쳐

                반면 SK하이닉스는 이 기간에 6.68% 하락했다. 골드만삭스가 투자의견을 '매수'에서 '중립'으로 하향 조정한 여파로 지난 17일 하루에만 8.95% 급락하기도 했다. 골드만삭스는 보고서를 통해 "2026년에 고대역폭메모리(HBM) 가격이 처음으로 하락할 수 있다"며 투자의견 조정 이유를 밝혔다.시장에선 올해 상반기 주가 상승세가 가팔랐던 SK하이닉스보단 삼성전자의 투자 매력도가 높다는 평가가 나온다. SK하이닉스는 상반기에 67.91% 올랐다. 반면 삼성전자 상승률은 12.41%에 그쳤다. 김용구 유안타증권(003470) 연구원은 "이제는 SK하이닉스를 팔고 삼성전자를 사는 게 맞다"며 "7월 말과 8월 초를 기점으로 반도체 전술적 대응 초점을 종전 SK하이닉스 매수(롱)·삼성전자 매도(쇼트) 주도에서 삼성전자 롱·

In [71]:
prompt = ChatPromptTemplate.from_template("""
* 다음은 사용자의 질문입니다:
{question}

* 관련된 문서:
{context}

                                        
최소 3개 이상 문서를 참고하여 사용자의 질문에 답변하세요.
너가 참고한 문서를 바탕으로, 아래 형식으로 답변해줘

* 질문에 대한 대답 형식
1) 대답 : 
2) 대답의 근거 : 
""")

In [74]:
llm = ChatOpenAI(api_key='')

rag_chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | llm

In [73]:
response = rag_chain.invoke({"question" :  "삼성전자 최근 3일치 주가는?"})
print(response.content)


1) 대답: 삼성전자의 최근 3일치 주가는 6만 7100원으로 상승한 것으로 나타났습니다.
2) 대답의 근거: 
- 삼성전자는 최근 3거래일 동안 상승세를 이어가며 연중 최고치를 경신했고, 전날까지 외국인 투자자들이 강력한 매수세를 보여주었습니다. (거시경제, 80fe7a8a-d597-4712-b855-f76e76f6f560)
- 이로 인해 삼성전자 주가는 최근 3일간 11.7% 상승하여 6만 7100원에 도달하였습니다. 또한, 삼성전자 주가가 7만원대에 안착할 것이라는 기대가 커졌습니다. (삼성전자, e72a5362-7326-47c4-8009-1487a64a1dd0)
- 외국인 투자자들의 강한 매수세와 기대감, 대법원의 무죄 판결로 인한 긍정적인 투심도 주가를 밀어올린 요인 중 하나입니다. (거시경제, edaff41d-7b6e-4edb-a8ca-da3148bad5ba)
